In [203]:
import os
import openai
import random
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai.api_key = os.getenv("OPENAI_API_KEY")


In [204]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_openai import ChatOpenAI
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage, AIMessage
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.schema.output_parser import StrOutputParser
from IPython.display import Image, display
from typing import Literal
from typing import Annotated, List
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.tools import tool
from pathlib import Path
from tempfile import TemporaryDirectory
from typing import Dict, Optional
from typing_extensions import TypedDict, List, Any
from typing import List, Optional, Literal
from langchain_core.language_models.chat_models import BaseChatModel
from langgraph.types import Command
from langgraph.prebuilt import create_react_agent

import json

In [205]:
class AgentState(TypedDict):
    input: HumanMessage
    messages: List[Any]
    corrigido: str
    decision: str

In [206]:
#tools
model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)



@tool
def read_file(arquivoteste: str) -> str:
    """Lê o conteúdo de um arquivo com base no nome do arquivo fornecido."""
    try:
        with open(arquivoteste, 'r') as file:
            return file.read()
    except FileNotFoundError:
        return "Arquivo não encontrado."
    
@tool
def write_file(arquivoteste: str, content: str) -> str:
    """"Escreve o conteúdo em um arquivo com base no nome do arquivo fornecido."""
    with open(arquivoteste, 'w') as file:
        file.write(content)
    return f"Conteúdo atualizado em {arquivoteste} com sucesso."


tools = [read_file, write_file]
llm_with_tools = model.bind_tools(tools)


In [ ]:
class Agent:
     def __init__(self, model ):
     
          self.model=model
          graph = StateGraph(AgentState)

          graph.add_node("Supervisor", self.supervisor)
          graph.add_node("Corretor", self.corretor)
          graph.add_node("Write", self.update_file)

          graph.set_entry_point("Supervisor")
          graph.add_conditional_edges("Supervisor", self.supervisor_router, {
                "Corretor": "Corretor",      
               "Write": "Write",  
            } )
          graph.add_edge("Corretor", END)
          graph.add_edge("Write", END)

          self.graph = graph.compile()
          
          

     def supervisor(self, state: AgentState):
          input_usuario = state['input']
          decision = self.model.invoke(f"""Você é um supervisor que delega tarefas com base em informações fornecidas. O usuário solicitou a seguinte ação: {input_usuario.content}.
          Se a ação envolver corrigir, revisar ou ajustar um arquivo, responda com "Corretor". Se a ação envolver escrever o conteúdo em um arquivo, responda com "Write". Responda apenas com uma das opções""")
          state ['decision'] = decision.content.strip()
          return state
     
     def supervisor_router(self, state: AgentState):
          return state['decision']


     def corretor(self, state: AgentState):
          input_usuario = state['input']
          tools = [read_file, write_file]
          llm_with_tools = self.model.bind_tools(tools)

          resposta = llm_with_tools.invoke([HumanMessage(content=f""" O usuário solicitou: {input_usuario.content} você tem a sua disposição as tools {tools} para te auxiliar""")])
          if resposta.tool_calls:
               results = []
               for call in resposta.tool_calls:
                    if call["name"] == "read_file":
                         conteudo = read_file.invoke(call["args"])
            
                         corrigido = self.model.invoke(f"Corrija este texto, não responda mais nada além da solicitação: {conteudo}")
                         results.append(AIMessage(content=corrigido.content))
                         state['corrigido'] = corrigido.content
                         nome_arquivo = call["args"]["arquivoteste"]
                         mensagem_final = write_file.invoke({
                    "arquivoteste": nome_arquivo, 
                    "content": corrigido.content})
                         results.append(AIMessage(content=mensagem_final))

          else:
               results = [AIMessage(content=resposta.content)]

          
          state['messages'] = state.get('messages', []) + results
          return state
     
          
          
          
     def update_file(self, state: AgentState):
          input_usuario = state['input']
          tools = [read_file, write_file]
          llm_with_tools = self.model.bind_tools(tools)
          resposta = llm_with_tools.invoke([HumanMessage(content=f""" O usuário solicitou: {input_usuario} você tem a sua disposição as tools {tools} para te auxiliar""")])
          results = []
          if resposta.tool_calls:
               for call in resposta.tool_calls:
                    if call["name"] == "write_file":
                         conteudo = write_file.invoke(call["args"])
                         results.append(AIMessage(content=conteudo))
          else:
               results = [AIMessage(content=resposta.content)]
            
                         
                    
          state['messages'] = state.get('messages', []) + results
          return state
     
         


         

In [208]:


model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)




In [209]:
if __name__ == "__main__":
    state = {
        'input': HumanMessage(content="arquivoteste.txt leia o texto do arquivo, e traduza em portugues."),
        'messages': [],
        'corrigido': '',
        'decision': '' 
    }

    agent = Agent(model)
    state_final = agent.graph.invoke(state)
    print("Estado final:", state_final['messages'][-1].content, state_final['decision'])

ValueError: Failed to reach https://mermaid.ink/ API while trying to render your graph. Status code: 502.

To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`